In [3]:
import os
import sys
from pathlib import Path
import torch
from transformers import BeitForImageClassification, BeitImageProcessor, BeitConfig
from huggingface_hub import snapshot_download, hf_hub_download
import json
from tqdm import tqdm
import time


In [4]:
# Download the BEIT model and processor


class ModelDownloader:
    def __init__(self, model_name, save_path):
        self.model_name = model_name
        self.save_path = Path(save_path)
        self.model_path = self.save_path / "beit-base-patch16-224"
        
    def create_directories(self):
        """Create necessary directories"""
        print(f"Creating directory structure at: {self.save_path}")
        self.save_path.mkdir(parents=True, exist_ok=True)
        self.model_path.mkdir(parents=True, exist_ok=True)
        print(f"✓ Directory created: {self.model_path}")
        
    def download_model_with_progress(self):
        """Download model with progress bar"""
        print(f"\n🚀 Starting download of {self.model_name}")
        print(f"📁 Saving to: {self.model_path}")
        
        try:
            # Download the entire repository with progress
            downloaded_path = snapshot_download(
                repo_id=self.model_name,
                cache_dir=str(self.model_path),
                local_dir=str(self.model_path),
                local_dir_use_symlinks=False,  # Create actual files, not symlinks
                resume_download=True,
                force_download=False  # Don't re-download if already exists
            )
            
            print(f"✓ Model downloaded successfully to: {downloaded_path}")
            return True
            
        except Exception as e:
            print(f"❌ Error downloading model: {str(e)}")
            return False
    
    def verify_download(self):
        """Verify that all necessary files are downloaded"""
        print(f"\n🔍 Verifying download...")
        
        required_files = [
            "config.json",
            "pytorch_model.bin",
            "preprocessor_config.json"
        ]
        
        missing_files = []
        existing_files = []
        
        for file in required_files:
            file_path = self.model_path / file
            if file_path.exists():
                existing_files.append(file)
                file_size = file_path.stat().st_size / (1024 * 1024)  # Size in MB
                print(f"  ✓ {file}: {file_size:.2f} MB")
            else:
                missing_files.append(file)
                print(f"  ❌ {file}: Missing")
        
        # Check for additional files
        all_files = list(self.model_path.glob("*"))
        additional_files = [f.name for f in all_files if f.name not in required_files and f.is_file()]
        
        if additional_files:
            print(f"\n📄 Additional files found:")
            for file in additional_files:
                file_path = self.model_path / file
                file_size = file_path.stat().st_size / (1024 * 1024)
                print(f"  • {file}: {file_size:.2f} MB")
        
        return len(missing_files) == 0, missing_files
    
    def load_and_test_model(self):
        """Load the model and processor to verify they work correctly"""
        print(f"\n🔧 Loading model for verification...")
        
        try:
            # Load processor
            print("Loading image processor...")
            processor = BeitImageProcessor.from_pretrained(str(self.model_path))
            print("✓ Image processor loaded successfully")
            
            # Load model
            print("Loading model...")
            model = BeitForImageClassification.from_pretrained(str(self.model_path))
            print("✓ Model loaded successfully")
            
            # Test model forward pass with dummy input
            print("Testing model with dummy input...")
            dummy_input = torch.randn(1, 3, 224, 224)  # Batch size 1, 3 channels, 224x224
            
            model.eval()
            with torch.no_grad():
                outputs = model(dummy_input)
                logits = outputs.logits
                print(f"✓ Model forward pass successful. Output shape: {logits.shape}")
            
            return True, model, processor
            
        except Exception as e:
            print(f"❌ Error loading model: {str(e)}")
            return False, None, None
    
    def get_model_summary(self, model, processor):
        """Generate a comprehensive model summary"""
        print(f"\n📊 MODEL SUMMARY")
        print("=" * 50)
        
        # Basic model info
        config = model.config
        print(f"Model Name: {self.model_name}")
        print(f"Architecture: {config.architectures[0] if hasattr(config, 'architectures') else 'BEIT'}")
        print(f"Model Type: {type(model).__name__}")
        
        # Model configuration
        print(f"\n🏗️  ARCHITECTURE:")
        print(f"  • Hidden Size: {config.hidden_size}")
        print(f"  • Number of Layers: {config.num_hidden_layers}")
        print(f"  • Number of Attention Heads: {config.num_attention_heads}")
        print(f"  • Intermediate Size: {config.intermediate_size}")
        print(f"  • Image Size: {config.image_size}")
        print(f"  • Patch Size: {config.patch_size}")
        print(f"  • Number of Channels: {config.num_channels}")
        
        # Classification info
        if hasattr(config, 'num_labels'):
            print(f"  • Number of Classes: {config.num_labels}")
        
        # Model parameters
        total_params = sum(p.numel() for p in model.parameters())
        trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        
        print(f"\n📈 PARAMETERS:")
        print(f"  • Total Parameters: {total_params:,}")
        print(f"  • Trainable Parameters: {trainable_params:,}")
        print(f"  • Model Size: ~{total_params * 4 / (1024**2):.1f} MB")
        
        # Processor info
        print(f"\n🖼️  IMAGE PROCESSOR:")
        if hasattr(processor, 'size'):
            print(f"  • Input Size: {processor.size}")
        if hasattr(processor, 'do_normalize') and processor.do_normalize:
            print(f"  • Normalization: Mean={processor.image_mean}, Std={processor.image_std}")
        if hasattr(processor, 'do_resize'):
            print(f"  • Resize: {processor.do_resize}")
        if hasattr(processor, 'do_center_crop'):
            print(f"  • Center Crop: {processor.do_center_crop}")
        
        # File information
        print(f"\n💾 STORAGE INFO:")
        total_size = sum(f.stat().st_size for f in self.model_path.rglob('*') if f.is_file())
        print(f"  • Total Download Size: {total_size / (1024**2):.1f} MB")
        print(f"  • Storage Location: {self.model_path}")
        print(f"  • Number of Files: {len(list(self.model_path.rglob('*')))}")
        
        # Usage recommendations
        print(f"\n💡 FINE-TUNING RECOMMENDATIONS:")
        print(f"  • Recommended Learning Rate: 1e-5 to 5e-5")
        print(f"  • Batch Size: 8-32 (depending on GPU memory)")
        print(f"  • Warm-up Steps: 10% of total training steps")
        print(f"  • Weight Decay: 0.01")
        print(f"  • Best for: Image classification tasks")
        
        return {
            'total_params': total_params,
            'trainable_params': trainable_params,
            'model_size_mb': total_params * 4 / (1024**2),
            'storage_size_mb': total_size / (1024**2),
            'config': config.to_dict() if hasattr(config, 'to_dict') else str(config)
        }
    
    def save_summary_to_file(self, summary_data):
        """Save model summary to a JSON file"""
        summary_file = self.model_path / "model_summary.json"
        
        # Convert any non-serializable objects to strings
        clean_summary = {}
        for key, value in summary_data.items():
            if isinstance(value, (int, float, str, list, dict)):
                clean_summary[key] = value
            else:
                clean_summary[key] = str(value)
        
        with open(summary_file, 'w', encoding='utf-8') as f:
            json.dump(clean_summary, f, indent=2, ensure_ascii=False)
        
        print(f"📄 Summary saved to: {summary_file}")

def main():
    # Configuration
    MODEL_NAME = "microsoft/beit-base-patch16-224-pt22k-ft22k"
    SAVE_PATH = "/Volumes/KODAK/folder_02/eye_disease_prediction_model/model/raw_model"
    
    print("🤖 BEIT Model Download and Verification Tool")
    print("=" * 60)
    
    # Initialize downloader
    downloader = ModelDownloader(MODEL_NAME, SAVE_PATH)
    
    try:
        # Step 1: Create directories
        downloader.create_directories()
        
        # Step 2: Download model
        download_success = downloader.download_model_with_progress()
        if not download_success:
            print("❌ Download failed. Exiting...")
            return
        
        # Step 3: Verify download
        verification_success, missing_files = downloader.verify_download()
        if not verification_success:
            print(f"❌ Verification failed. Missing files: {missing_files}")
            return
        else:
            print("✓ All required files downloaded successfully!")
        
        # Step 4: Load and test model
        load_success, model, processor = downloader.load_and_test_model()
        if not load_success:
            print("❌ Model loading failed. Exiting...")
            return
        
        # Step 5: Generate summary
        summary_data = downloader.get_model_summary(model, processor)
        
        # Step 6: Save summary
        downloader.save_summary_to_file(summary_data)
        
        print(f"\n🎉 SUCCESS! Model is ready for fine-tuning!")
        print(f"📍 Model location: {downloader.model_path}")
        print(f"\n🚀 Next steps:")
        print(f"   1. Prepare your eye disease dataset")
        print(f"   2. Set up data preprocessing")
        print(f"   3. Configure training parameters")
        print(f"   4. Start fine-tuning!")
        
    except KeyboardInterrupt:
        print(f"\n\n⚠️  Download interrupted by user")
    except Exception as e:
        print(f"\n❌ Unexpected error: {str(e)}")
        print(f"Please check your internet connection and try again.")

if __name__ == "__main__":
    main()

🤖 BEIT Model Download and Verification Tool
Creating directory structure at: /Volumes/KODAK/folder_02/eye_disease_prediction_model/model/raw_model
✓ Directory created: /Volumes/KODAK/folder_02/eye_disease_prediction_model/model/raw_model/beit-base-patch16-224

🚀 Starting download of microsoft/beit-base-patch16-224-pt22k-ft22k
📁 Saving to: /Volumes/KODAK/folder_02/eye_disease_prediction_model/model/raw_model/beit-base-patch16-224


/Volumes/KODAK/folder_02/eye_disease_prediction_model/venv/lib/python3.12/site-packages/huggingface_hub/file_download.py:980: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(
/Volumes/KODAK/folder_02/eye_disease_prediction_model/venv/lib/python3.12/site-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

README.md: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/276 [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/414M [00:00<?, ?B/s]

flax_model.msgpack:   0%|          | 0.00/410M [00:00<?, ?B/s]

✓ Model downloaded successfully to: /Volumes/KODAK/folder_02/eye_disease_prediction_model/model/raw_model/beit-base-patch16-224

🔍 Verifying download...
  ✓ config.json: 1.59 MB
  ✓ pytorch_model.bin: 394.87 MB
  ✓ preprocessor_config.json: 0.00 MB

📄 Additional files found:
  • ._models--microsoft--beit-base-patch16-224-pt22k-ft22k: 0.00 MB
  • ._.cache: 0.00 MB
  • README.md: 0.01 MB
  • ._README.md: 0.00 MB
  • ._preprocessor_config.json: 0.00 MB
  • .gitattributes: 0.00 MB
  • ._.gitattributes: 0.00 MB
  • ._config.json: 0.00 MB
  • ._pytorch_model.bin: 0.00 MB
  • flax_model.msgpack: 391.23 MB
  • ._flax_model.msgpack: 0.00 MB
✓ All required files downloaded successfully!

🔧 Loading model for verification...
Loading image processor...
✓ Image processor loaded successfully
Loading model...


/Volumes/KODAK/folder_02/eye_disease_prediction_model/venv/lib/python3.12/site-packages/transformers/utils/deprecation.py:172: UserWarning: The following named arguments are not valid for `BeitImageProcessor.__init__` and were ignored: 'feature_extractor_type'
  return func(*args, **kwargs)


✓ Model loaded successfully
Testing model with dummy input...
✓ Model forward pass successful. Output shape: torch.Size([1, 21841])

📊 MODEL SUMMARY
Model Name: microsoft/beit-base-patch16-224-pt22k-ft22k
Architecture: BeitForImageClassification
Model Type: BeitForImageClassification

🏗️  ARCHITECTURE:
  • Hidden Size: 768
  • Number of Layers: 12
  • Number of Attention Heads: 12
  • Intermediate Size: 3072
  • Image Size: 224
  • Patch Size: 16
  • Number of Channels: 3
  • Number of Classes: 21841

📈 PARAMETERS:
  • Total Parameters: 102,557,713
  • Trainable Parameters: 102,557,713
  • Model Size: ~391.2 MB

🖼️  IMAGE PROCESSOR:
  • Input Size: {'height': 224, 'width': 224}
  • Normalization: Mean=[0.5, 0.5, 0.5], Std=[0.5, 0.5, 0.5]
  • Resize: True
  • Center Crop: False

💾 STORAGE INFO:
  • Total Download Size: 787.8 MB
  • Storage Location: /Volumes/KODAK/folder_02/eye_disease_prediction_model/model/raw_model/beit-base-patch16-224
  • Number of Files: 50

💡 FINE-TUNING RECOMMEN